In [1]:
import os
import numpy as np
import scipy as sp
from src.data import DataLoader
from src.timescales import Timescales

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
dataset = 'HCP-YA'
data_loader = DataLoader(dataset=dataset)

Initialized DataLoader for dataset: HCP-YA
Base directory: /mnt/storage_ssd_raid/research_data/HCP_YA
Raw data directory: /mnt/storage_ssd_raid/research_data/HCP_YA/HCP_1200
Derivatives data directory: /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives
Output directory: /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales
Loaded behavioral data: 1206 subjects


# fMRI data

In [3]:
subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids.txt'), dtype=str)
# subject_ids = subject_ids[:10]
subject_ids

array(['100206', '100307', '100408', ..., '994273', '995174', '996782'],
      dtype='<U6')

In [4]:
# Register 3T rest scans
rest_scans = data_loader.register_3t_rest_scans(
    subject_ids=subject_ids,
    atlas='Schaefer4007',
)

# Register 3T task scans
task_scans = data_loader.register_3t_task_scans(
    subject_ids=subject_ids,
    atlas='Schaefer4007',
)

# Register 7T rest scans
task_scans = data_loader.register_7t_rest_scans(
    subject_ids=subject_ids,
    atlas='Schaefer4007',
)

# Register 7T movie scans
task_scans = data_loader.register_7t_movie_scans(
    subject_ids=subject_ids,
    atlas='Schaefer4007',
)

In [5]:
_ = data_loader.validate_scan_registry()

Validating 26712 scans in registry...
  Missing: sub-101410_modality-MRI3T_tr-0.72_condition-rest_scan-rfMRIREST1LR_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/fmri_parcellated/101410/101410_task-rfMRIREST1LR_atlas-Schaefer4007_stat-mean_timeseries.ptseries.nii
  Missing: sub-101410_modality-MRI3T_tr-0.72_condition-rest_scan-rfMRIREST1RL_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/fmri_parcellated/101410/101410_task-rfMRIREST1RL_atlas-Schaefer4007_stat-mean_timeseries.ptseries.nii
  Missing: sub-101410_modality-MRI3T_tr-0.72_condition-rest_scan-rfMRIREST2LR_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/fmri_parcellated/101410/101410_task-rfMRIREST2LR_atlas-Schaefer4007_stat-mean_timeseries.ptseries.nii
  Missing: sub-101410_modality-MRI3T_tr-0.72_condition-rest_scan-rfMRIREST2RL_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/fmri_parcellated/101410/101410_task-

/home/lindenmp/anaconda3/envs/intrinsic_neural_timescales/lib/python3.11/site-packages/src/data.py:870: UserWarning: genfromtxt: Empty input file: "/mnt/storage_ssd_raid/research_data/HCP_YA/HCP_1200/196952/MNINonLinear/Results/tfMRI_WM_LR/EVs/0bk_places.txt"
  ev_data = np.genfromtxt(os.path.join(ev_dir, condition))


  Missing: sub-199251_modality-MRI3T_tr-0.72_condition-task_scan-tfMRIWMLR_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/fmri_parcellated/199251/199251_task-tfMRIWMLR_atlas-Schaefer4007_stat-mean_timeseries.ptseries.nii
  Missing: sub-199251_modality-MRI3T_tr-0.72_condition-task_scan-tfMRIWMRL_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/fmri_parcellated/199251/199251_task-tfMRIWMRL_atlas-Schaefer4007_stat-mean_timeseries.ptseries.nii
  Missing: sub-199251_modality-MRI3T_tr-0.72_condition-task_scan-tfMRIEMOTIONLR_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/fmri_parcellated/199251/199251_task-tfMRIEMOTIONLR_atlas-Schaefer4007_stat-mean_timeseries.ptseries.nii
  Missing: sub-199251_modality-MRI3T_tr-0.72_condition-task_scan-tfMRIEMOTIONRL_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/fmri_parcellated/199251/199251_task-tfMRIEMOTIONRL_atlas-Schaefer4007_stat-mean_

In [6]:
data_loader.export_scan_registry('scan_registry_fmri.csv')
scan_registry = data_loader.list_registered_scans()
scan_registry.head()

,scan_string,subject_id,modality,scan,atlas,condition,tr
0,sub-100206_modality-MRI3T_tr-0.72_condition-re...,100206,MRI3T,rfMRIREST1LR,Schaefer4007,rest,0.72
1,sub-100206_modality-MRI3T_tr-0.72_condition-re...,100206,MRI3T,rfMRIREST1RL,Schaefer4007,rest,0.72
2,sub-100206_modality-MRI3T_tr-0.72_condition-re...,100206,MRI3T,rfMRIREST2LR,Schaefer4007,rest,0.72
3,sub-100206_modality-MRI3T_tr-0.72_condition-re...,100206,MRI3T,rfMRIREST2RL,Schaefer4007,rest,0.72
4,sub-100307_modality-MRI3T_tr-0.72_condition-re...,100307,MRI3T,rfMRIREST1LR,Schaefer4007,rest,0.72


In [7]:
subject_ids = scan_registry['subject_id'].unique()
n_subs = len(subject_ids)
print(n_subs)

982


In [8]:
# Load the data
fmri_data = data_loader.load_batch(
    nuisance_regression=True,
    verbose=True
)

print(f"Loaded {len(fmri_data)} scans")
print(fmri_data.keys())

Loading 16922 scans...
  [1/16922] 100206 rfMRIREST1LR
  [11/16922] 100408 rfMRIREST2LR
  [21/16922] 101107 rfMRIREST1LR
  [31/16922] 101915 rfMRIREST2LR
  [41/16922] 102311 rfMRIREST1LR
  [51/16922] 102614 rfMRIREST2LR
  [61/16922] 103111 rfMRIREST1LR
  [71/16922] 103414 rfMRIREST2LR
  [81/16922] 104012 rfMRIREST1LR
  [91/16922] 104820 rfMRIREST2LR
  [101/16922] 105216 rfMRIREST1LR
  [111/16922] 105923 rfMRIREST2LR
  [121/16922] 106521 rfMRIREST1LR
  [131/16922] 107018 rfMRIREST2LR
  [141/16922] 107725 rfMRIREST1LR
  [151/16922] 108121 rfMRIREST2LR
  [161/16922] 108525 rfMRIREST1LR
  [171/16922] 109123 rfMRIREST2LR
  [181/16922] 110411 rfMRIREST1LR
  [191/16922] 111009 rfMRIREST2LR
  [201/16922] 111413 rfMRIREST1LR
  [211/16922] 111716 rfMRIREST2LR
  [221/16922] 112516 rfMRIREST1LR
  [231/16922] 113215 rfMRIREST2LR
  [241/16922] 113922 rfMRIREST1LR
  [251/16922] 114318 rfMRIREST2LR
  [261/16922] 114823 rfMRIREST1LR
  [271/16922] 115219 rfMRIREST2LR
  [281/16922] 115825 rfMRIREST1LR
  

## Process fMRI timescales

In [9]:
timescales_processor = Timescales(dataset=dataset, timeseries_data=fmri_data, verbose=True)

Initialized DataLoader for dataset: HCP-YA
Base directory: /mnt/storage_ssd_raid/research_data/HCP_YA
Raw data directory: /mnt/storage_ssd_raid/research_data/HCP_YA/HCP_1200
Derivatives data directory: /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives
Output directory: /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales
Loaded behavioral data: 1206 subjects

	Initialized Timescales for dataset: HCP-YA
	Number of scans: 16922
	Output directory: /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales


In [10]:
timescale_measures = [
    'zero_crossing',
    'auc_zero_crossing'
]

In [11]:
timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=False, verbose=True)
for timescale_measure in timescale_measures:
    timescales_processor.compute_timescale_measures(method=timescale_measure, verbose=True)

Computing autocorrelation: 100%|██████████| 16922/16922 [05:17<00:00, 53.29it/s]



Autocorrelation computation complete:
  Loaded from cache: 0
  Newly computed: 16922
  Total: 16922


Computing zero_crossing timescales: 100%|██████████| 16922/16922 [00:14<00:00, 1132.20it/s]



zero_crossing timescale computation complete:
  Loaded from cache: 0
  Newly computed: 16922
  Total: 16922


Computing auc_zero_crossing timescales: 100%|██████████| 16922/16922 [00:35<00:00, 478.19it/s]


auc_zero_crossing timescale computation complete:
  Loaded from cache: 0
  Newly computed: 16922
  Total: 16922


In [12]:
timescales_processor.regress_task_structure()

Regressing task structure out of timeseries: 100%|██████████| 16922/16922 [04:57<00:00, 56.90it/s]  


In [13]:
timescales_processor.compute_autocorrelation_function(use_fft=True, use_residuals=True, verbose=True)
for timescale_measure in timescale_measures:
    timescales_processor.compute_timescale_measures(method=timescale_measure, suffix='resid', verbose=True)

Computing autocorrelation: 100%|██████████| 16922/16922 [05:30<00:00, 51.19it/s]



Autocorrelation computation complete:
  Loaded from cache: 0
  Newly computed: 16922
  Total: 16922


Computing zero_crossing timescales: 100%|██████████| 16922/16922 [00:14<00:00, 1164.21it/s]



zero_crossing timescale computation complete:
  Loaded from cache: 0
  Newly computed: 16922
  Total: 16922


Computing auc_zero_crossing timescales: 100%|██████████| 16922/16922 [00:34<00:00, 483.54it/s]


auc_zero_crossing timescale computation complete:
  Loaded from cache: 0
  Newly computed: 16922
  Total: 16922
